# Part 1: rag chain for bash query

In [397]:
import os
import json
import logging
from pathlib import Path

def load_env_file(path):
    if not path.exists():
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

# Load .env from same directory as script
env_path = Path.cwd() / ".env"
load_env_file(env_path)

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

# Verify API key is set
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
elif os.getenv("OPENAI_API_KEY"):
    logger.info("OPENAI_API_KEY is set")
elif os.getenv("ANTHROPIC_API_KEY"):
    logger.info("ANTHROPIC_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

# Check for Cohere API key (needed for re-ranking)
if os.getenv("COHERE_API_KEY"):
    logger.info("COHERE_API_KEY is set (for re-ranking)")
else:
    logger.warning("COHERE_API_KEY not set - re-ranking section will not work")



[2026-03-01 19:07:58,776] p93245 {3678128238.py:30} INFO - GROQ_API_KEY is set
[2026-03-01 19:07:58,783] p93245 {3678128238.py:40} INFO - COHERE_API_KEY is set (for re-ranking)


In [398]:
from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
# You can change this to other models like:
# - "groq/llama-3.3-70b-versatile" (larger, FREE)
# - "gpt-4o-mini" (OpenAI, paid)
# - "claude-3-5-haiku-20241022" (Anthropic, paid)

MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

from langchain_community.embeddings import HuggingFaceEmbeddings

# Using a free, local embedding model
embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
logger.info("Embeddings model loaded successfully")

[2026-03-01 19:08:04,563] p93245 {480159892.py:15} INFO - Using model: groq/llama-3.1-8b-instant
[2026-03-01 19:08:04,622] p93245 {SentenceTransformer.py:219} INFO - Use pytorch device_name: mps
[2026-03-01 19:08:04,622] p93245 {SentenceTransformer.py:227} INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1035.01it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[2026-03-01 19:08:07,473] p93245 {480159892.py:23} INFO - Embeddings model loaded successfully


### step 1: classify

In [261]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# step 1: classify
classify_template = (
    "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    "Classify the incoming question to determine what type of information is needed. "
    "Then provide several specific file name as advice to guide further search. "
    "Respond with ONE short sentence for each, concise.\n\n"
    "Question: {input}"
)
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", classify_template),
])

classifier_chain = classify_prompt | llm | StrOutputParser()

In [262]:
question = "What Python dependencies does this project use?"
label = classifier_chain.invoke({"input": question})
print(label)

20:04:26 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 20:04:26,084] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:04:26 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 20:04:26,375] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


The question pertains to project dependencies, specifically Python packages.

Advice to guide further search:
- Check the 'requirements.txt' file.
- Look into the 'setup.py' file.
- Search for 'pip freeze' output in the 'README.md' file.
- Inspect the 'pyproject.toml' file.
- Review the 'Pipfile' file if using Pipenv.


### step 2: plan and select bash tool (high level)

In [270]:
import subprocess

# step 2: plan search

tree = subprocess.run("tree .", shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
select_template = (
    "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    "Plan to select the appropriate bash tool(s) to have overall knowledge of sturcture to further retrieve relevant context base on classified intent, strategy advice and basic dataset structure"
    "In current step, search the directory first with bash commands for overall knowledge of the dataset for further step base on question and intent, DO NOT directly cat any file yet!"
    "Return JSON ONLY, no markdown, no backticks, no commentary\n"
    "Schema:\n"
    '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
    "Constraints:\n"
    "- cmds must be an array of strings\n"
    "- 1 to 10 commands only\n"
    "- No cat in this step\n"
    "- Each command must end with: | head -n 100\n"
    "- Prefer find/rg/grep/ls; restrict file types when possible\n\n"
    "Rules:\n"
    "Consider understanding the overall structure of directory "
    "Consider grep, rg, find, ls, cat, head, tail\n"
    "Do not use -l when ls\n"
    #"- Use grep with -rni -C 3 if possible\n"
    "- Prefer restricting file types with --include when possible\n"
    "- End each command with | head -n 100\n\n"
    "Question: {input}\n"
    "Intention: {intent}\n"
    "Data structure:{structure}\n"
)
select_prompt = ChatPromptTemplate.from_messages([
    ("system", select_template),
])

select_chain = select_prompt | llm | StrOutputParser()

In [272]:
cmds_text = select_chain.invoke({
    "input": question,
    "intent": label,
    "structure": tree.stdout,
})
print(cmds_text)

20:15:44 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 20:15:44,816] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:15:46 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 20:15:46,338] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


{
  "cmds": [
    "find . -type f -name requirements.txt | head -n 100",
    "find . -type f -name setup.py | head -n 100",
    "rg 'pip freeze' README.md | head -n 100",
    "find . -type f -name pyproject.toml | head -n 100",
    "find . -type f -name Pipfile | head -n 100"
  ]
}


### step 3: execute bash commands (high level)

In [273]:
import re

import json

ALLOWED_PREFIXES = ("ls ", "find ", "rg ", "grep ", "cat ", "sed ", "awk ", "head ", "tail ")

def extract_commands(text: str) -> list[str]:
    """
    Expect JSON: {"cmds": ["...", "..."]}
    Returns a filtered list of allowed shell commands.
    """
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        # Fallback: sometimes the model adds extra text—try to extract the first JSON object
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return []
        data = json.loads(text[start : end + 1])

    cmds = data.get("cmds", [])
    if not isinstance(cmds, list):
        return []

    cleaned: list[str] = []
    for c in cmds:
        if not isinstance(c, str):
            continue
        c = c.strip()
        if not c:
            continue
        if c.startswith(ALLOWED_PREFIXES) or c in ("ls", "pwd", "whoami"):
            cleaned.append(c)

    return cleaned


commands = extract_commands(cmds_text)
commands

['find . -type f -name requirements.txt | head -n 100',
 'find . -type f -name setup.py | head -n 100',
 "rg 'pip freeze' README.md | head -n 100",
 'find . -type f -name pyproject.toml | head -n 100',
 'find . -type f -name Pipfile | head -n 100']

In [278]:
import subprocess

high_level_result = ""
for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
    # print(result.stdout)
    high_level_result += "Ran: " + cmd + "\n" + "Found: " + result.stdout + "\n\n"

In [279]:
high_level_result

"Ran: find . -type f -name requirements.txt | head -n 100\nFound: ./terraform/aws-ecs/scripts/requirements.txt\n\n\nRan: find . -type f -name setup.py | head -n 100\nFound: \n\nRan: rg 'pip freeze' README.md | head -n 100\nFound: \n\nRan: find . -type f -name pyproject.toml | head -n 100\nFound: ./auth_server/pyproject.toml\n./pyproject.toml\n./agents/a2a/pyproject.toml\n./metrics-service/pyproject.toml\n./servers/example-server/pyproject.toml\n./servers/currenttime/pyproject.toml\n./servers/fininfo/pyproject.toml\n./servers/mcpgw/pyproject.toml\n./servers/realserverfaketools/pyproject.toml\n\n\nRan: find . -type f -name Pipfile | head -n 100\nFound: \n\n"

### step 4: select bash (low level extract)

In [280]:
# step 4: search
select_template_2 = (
    "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    "Plan to select the appropriate bash tool(s) to find specific files base on classified intent and given overall bash search record for further read"
    "Return JSON ONLY, no markdown, no backticks, no commentary\n"
    "Schema:\n"
    '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
    "Constraints:\n"
    "- cmds must be an array of strings\n"
    "- 1 to 10 commands only\n"
    "- No cat in this step\n"
    "- Each command must end with: | head -n 100\n"
    "Rules:\n"
    "Use -rHn -C 2 if use grep\n"
    "- Prefer restricting file types with --include when possible\n"
    "Question: {input}\n"
    "Intention: {intent}\n"
    "Record: {record}\n"
)
select_prompt_l = ChatPromptTemplate.from_messages([
    ("system", select_template_2),
])

select_chain_lower = select_prompt_l | llm | StrOutputParser()

In [281]:
cmds_text_lower = select_chain_lower.invoke({
    "input": question,
    "intent": label,
    "record": high_level_result,
})
print(cmds_text_lower)

20:23:20 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 20:23:20,483] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:23:20 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 20:23:20,853] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


{"cmds": ["rg 'pip freeze' README.md -rHn -C 2 | head -n 100", "find . -type f -name requirements.txt | head -n 100", "find . -type f -name setup.py | head -n 100", "find . -type f -name pyproject.toml | head -n 100", "find . -type f -name Pipfile | head -n 100"]}


### step 5: execute bash (low level)

In [282]:
commands_2 = extract_commands(cmds_text_lower)
commands_2

["rg 'pip freeze' README.md -rHn -C 2 | head -n 100",
 'find . -type f -name requirements.txt | head -n 100',
 'find . -type f -name setup.py | head -n 100',
 'find . -type f -name pyproject.toml | head -n 100',
 'find . -type f -name Pipfile | head -n 100']

In [285]:
import subprocess

context_result = ""
for cmd in commands_2:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
    # print(result.stdout)
    context_result += "Ran: " + cmd + "\n" + "Found: " + result.stdout + "\n\n"

In [286]:
context_result

"Ran: rg 'pip freeze' README.md -rHn -C 2 | head -n 100\nFound: \n\nRan: find . -type f -name requirements.txt | head -n 100\nFound: ./terraform/aws-ecs/scripts/requirements.txt\n\n\nRan: find . -type f -name setup.py | head -n 100\nFound: \n\nRan: find . -type f -name pyproject.toml | head -n 100\nFound: ./auth_server/pyproject.toml\n./pyproject.toml\n./agents/a2a/pyproject.toml\n./metrics-service/pyproject.toml\n./servers/example-server/pyproject.toml\n./servers/currenttime/pyproject.toml\n./servers/fininfo/pyproject.toml\n./servers/mcpgw/pyproject.toml\n./servers/realserverfaketools/pyproject.toml\n\n\nRan: find . -type f -name Pipfile | head -n 100\nFound: \n\n"

### step 6: pass and open

In [295]:
# step 6: open
select_template_3 = (
    "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    "Plan to use command 'cat' to print at most 10 relevant files for context base on classified intent and given bash search record"
    "Return JSON ONLY, no markdown, no backticks, no commentary\n"
    "Schema:\n"
    '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
    "Constraints:\n"
    "- cmds must be an array of strings\n"
    "- 1 to 10 commands only\n"
    "- Use cat in this step\n"
    "Rules:\n"
    "Consider cat command ONLY\n"
    "Question: {input}\n"
    "Intention: {intent}\n"
    "Record: {record}\n"
)
select_prompt_c = ChatPromptTemplate.from_messages([
    ("system", select_template_3),
])

select_chain_open = select_prompt_c | llm | StrOutputParser()

In [296]:
cmds_text_open = select_chain_open.invoke({
    "input": question,
    "intent": label,
    "record": context_result,
})

print(cmds_text_open)

20:27:27 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 20:27:27,585] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:27:27 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 20:27:27,897] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


{"cmds": ["cat ./terraform/aws-ecs/scripts/requirements.txt", "cat ./auth_server/pyproject.toml", "cat ./pyproject.toml", "cat ./agents/a2a/pyproject.toml", "cat ./metrics-service/pyproject.toml", "cat ./servers/example-server/pyproject.toml", "cat ./servers/currenttime/pyproject.toml", "cat ./servers/fininfo/pyproject.toml", "cat ./servers/mcpgw/pyproject.toml", "cat ./servers/realserverfaketools/pyproject.toml"]}


### step 7: execute bash (open)

In [297]:
commands_3 = extract_commands(cmds_text_open)
commands_3

['cat ./terraform/aws-ecs/scripts/requirements.txt',
 'cat ./auth_server/pyproject.toml',
 'cat ./pyproject.toml',
 'cat ./agents/a2a/pyproject.toml',
 'cat ./metrics-service/pyproject.toml',
 'cat ./servers/example-server/pyproject.toml',
 'cat ./servers/currenttime/pyproject.toml',
 'cat ./servers/fininfo/pyproject.toml',
 'cat ./servers/mcpgw/pyproject.toml',
 'cat ./servers/realserverfaketools/pyproject.toml']

In [298]:
import subprocess

context_open = ""
for cmd in commands_3:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
    # print(result.stdout)
    context_open +=  "Ran: " + cmd + "\n" + "Found: "+  result.stdout + "\n------------\n"


In [299]:
context_open

'Ran: cat ./terraform/aws-ecs/scripts/requirements.txt\nFound: pydantic>=2.0.0\nrequests>=2.31.0\n\n------------\nRan: cat ./auth_server/pyproject.toml\nFound: [build-system]\nrequires = ["setuptools>=42.0", "wheel"]\nbuild-backend = "setuptools.build_meta"\n\n[tool.setuptools]\npackages = ["auth_server"]\n\n[project]\nname = "auth_server"\nversion = "0.1.0"\ndescription = "Authentication server for validating JWT tokens against Amazon Cognito"\nrequires-python = ">=3.9"\ndependencies = [\n    "fastapi>=0.115.0",\n    "uvicorn[standard]>=0.34.0",\n    "pydantic>=2.0.0",\n    "pydantic-settings>=2.0.0",\n    "requests>=2.28.0",\n    "python-jose>=3.3.0",\n    "python-dotenv>=1.0.0",\n    "boto3>=1.28.0",\n    "pyjwt>=2.6.0",\n    "cryptography>=40.0.0",\n    "pyyaml>=6.0.0",\n    "httpx>=0.25.0",\n    "itsdangerous>=2.1.0",\n    "opensearch-py>=2.4.0",\n    "aiohttp>=3.8.0",\n    "motor>=3.3.0",\n    "pymongo>=4.6.0",\n    "aiofiles>=24.1.0"\n]\n\n[project.optional-dependencies]\ndev = 

### step 8: generate

In [300]:
# step 8: generate
generate_templete = (
    "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    "Use record of grep search as dataset to answer following question, say 'I don't know' if there is no relavent info in dataset"
    "Question: {input}\n"
    "Record: {record}\n"
)
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", generate_templete),
])

generate_chain = generate_prompt | llm | StrOutputParser()

In [301]:
generate_result = generate_chain.invoke({
    "input": question,
    "record": context_open,
})
print(generate_result)

20:27:45 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 20:27:45,603] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
20:27:48 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 20:27:48,169] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Based on the provided records, the Python dependencies used by this project are:

- `pydantic>=2.0.0`
- `requests>=2.31.0`
- `fastapi>=0.115.0`
- `uvicorn[standard]>=0.34.0`
- `pydantic-settings>=2.0.0`
- `python-jose>=3.3.0`
- `python-dotenv>=1.0.0`
- `boto3>=1.28.0`
- `pyjwt>=2.6.0`
- `cryptography>=40.0.0`
- `pyyaml>=6.0.0`
- `httpx>=0.25.0`
- `itsdangerous>=2.1.0`
- `opensearch-py>=2.4.0`
- `aiohttp>=3.8.0`
- `motor>=3.3.0`
- `pymongo>=4.6.0`
- `aiofiles>=24.1.0`
- `rich>=13.0.0`
- `cisco-ai-a2a-scanner @ git+https://github.com/cisco-ai-defense/a2a-scanner.git@main`
- `cisco-ai-skill-scanner>=1.0.0`
- `awscli>=1.36.0`
- `litellm>=1.50.0`
- `faiss-cpu>=1.7.4`
- `sentence-transformers>=3.0.0`
- `websockets>=15.0.1`
- `scikit-learn>=1.3.0`
- `torch>=1.6.0`
- `huggingface-hub>=0.31.1`
- `bandit>=1.8.3`
- `langchain-mcp-adapters>=0.0.11`
- `langgraph>=0.4.3`
- `langchain-aws>=0.2.23`
- `pytz>=2025.2`
- `strands-agents>=0.1.6`
- `strands-agents-tools>=0.1.4`
- `httpcore[asyncio]>=1.0.9`


## Wrap together

In [388]:
    
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import subprocess
import json

DANGEROUS_TOKENS = (";", "&&", "||", ">", ">>", "<", "$(", "`", "| sh", "|bash", "rm ", "mv ", "chmod ", "chown ")

def is_safe(cmd: str) -> bool:
    c = cmd.strip().lower()
    head = c.split(maxsplit=1)[0] if c else ""
    if head not in {"ls","find","rg","grep","cat","sed","awk","head","tail","tree","pwd","whoami"}:
        return False
    return not any(tok in c for tok in DANGEROUS_TOKENS)

ALLOWED_PREFIXES = ("ls ", "find ", "rg ", "grep ", "cat ", "sed ", "awk ", "head ", "tail ")

def _repair_invalid_backslashes(s):
    """

    Any backslash followed by something else must be doubled.
    """
    # Replace \X where X is NOT valid JSON escape char
    return re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', s)

def extract_commands(text: str) -> list[str]:
    """
    Expect JSON: {"cmds": ["...", "..."]}
    Returns a filtered list of allowed shell commands.
    """
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return []

        chunk = text[start:end+1]

        repaired = _repair_invalid_backslashes(chunk)
        try:
            data = json.loads(repaired)
        except json.JSONDecodeError:
            return []

    cmds = data.get("cmds", [])
    if not isinstance(cmds, list):
        return []

    cleaned: list[str] = []
    for c in cmds:
        if not isinstance(c, str):
            continue
        c = c.strip()
        if not c:
            continue
        if c.startswith(ALLOWED_PREFIXES) or c in ("ls", "pwd", "whoami"):
            cleaned.append(c)

    return cleaned

def limit_commands(text, max_cmds=5):
    cmds = [c.strip() for c in text.split("\n\n") if c.strip()]
    return "\n\n".join(cmds[:max_cmds])



def mcp_qa_agent(question):
    
    # step 1: classify
    classify_template = (
        "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
        "Classify the incoming question to determine what type of information is needed. "
        "Then provide several specific file name as advice to guide further search. "
        "Respond with ONE short sentence for each, concise.\n\n"
        "Question: {input}"
    )
    classify_prompt = ChatPromptTemplate.from_messages([
        ("system", classify_template),
    ])
    classifier_chain = classify_prompt | llm | StrOutputParser()
    label = classifier_chain.invoke({"input": question})

    # step 2: plan search
    tree = subprocess.run("tree .", shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
    select_template = (
        "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
        "Plan to select the appropriate bash tool(s) to have overall knowledge of sturcture to further retrieve relevant context base on classified intent, strategy advice and basic dataset structure"
        "DO NOT directly cat any file yet!"
        "Return JSON ONLY, no markdown, no backticks, no commentary\n"
        "Schema:\n"
        '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
        "Constraints:\n"
        "- Do not use any backslash characters in any command."
        "- cmds must be an array of strings\n"
        "- 1 to 8 commands only\n"
        "- No cat in this step\n"
        "- Each command must end with: | head -n 100\n"
        "- Prefer find/rg/grep/ls; restrict file types when possible\n\n"
        "Rules:\n"
        "Consider understanding the overall structure of directory "
        "Consider grep, rg, find, ls, cat, head, tail\n"
        "Do not use -l when ls\n"
        #"- Use grep with -rni -C 3 if possible\n"
        "- Prefer restricting file types with --include when possible\n"
        "- End each command with | head -n 100\n\n"
        "Question: {input}\n"
        "Intention: {intent}\n"
        "Data structure:{structure}\n"
    )
    select_prompt = ChatPromptTemplate.from_messages([
        ("system", select_template),
    ])

    select_chain = select_prompt | llm | StrOutputParser()
    cmds_text = select_chain.invoke({
        "input": question,
        "intent": label,
        "structure": tree.stdout
    })

    # step 3: execute command (higher)
    commands = extract_commands(cmds_text)
    commands = [c for c in commands if is_safe(c)]
    high_level_result = ""
    for cmd in commands:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
        # print(result.stdout)
        high_level_result += "Ran: " + cmd + "\n" + "Found: " + result.stdout + "\n\n"

    # # step 4: search
    # select_template_2 = (
    #     "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
    #     "Plan to select the appropriate bash tool(s) to find specific files base on classified intent, dataset structure and given overall bash search record for further read"
    #     "You MUST return JSON ONLY, no markdown, no backticks, no commentary\n"
    #     "Schema:\n"
    #     '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
    #     "Constraints:\n"
    #     "- Do not use any backslash characters in any command."
    #     "- cmds must be an array of strings\n"
    #     "- 1 to 10 commands only\n"
    #     "- No cat in this step\n"
    #     "- Each command must end with: | head -n 100\n"
    #     "Rules:\n"
    #     "Use -rHn -C 2 if use grep\n"
    #     "- Prefer restricting file types with --include when possible\n"
    #     "Question: {input}\n"
    #     "Intention: {intent}\n"
    #     "Record: {record}\n"
    #     "Structure: {structure}\n"
    # )
    # select_prompt_l = ChatPromptTemplate.from_messages([
    #     ("system", select_template_2),
    # ])
    # select_chain_lower = select_prompt_l | llm | StrOutputParser()
    # cmds_text_lower = select_chain_lower.invoke({
    #     "input": question,
    #     "intent": label,
    #     "record": high_level_result,
    #     "structure": tree,
    # })

    # # step 5: execute command (lower)
    # commands_2 = extract_commands(cmds_text_lower)
    # commands_2 = [c for c in commands_2 if is_safe(c)]
    # context_result = ""
    # for cmd in commands_2:
    #     result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
    #     # print(result.stdout)
    #     context_result += "Ran: " + cmd + "\n" + "Found: " + result.stdout + "\n\n"

    # step 6: open
    select_template_3 = (
        "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
        "Plan to use command 'cat' to print at most 10 relevant files for context base on classified intent, dataset structure and given bash search record"
        "You MUST return JSON ONLY, no markdown, no backticks, no commentary\n"
        "Schema:\n"
        '{{ "cmds": ["<shell command 1>", "<shell command 2>"] }}\n\n'
        "Constraints:\n"
        "- Do not use any backslash characters in any command."
        "- cmds must be an array of strings\n"
        "- 1 to 8 commands only\n"
        "- Use cat in this step\n"
        "- Each command must end with: | head -n 300\n"
        "Rules:\n"
        "Consider cat command ONLY\n"
        "Question: {input}\n"
        "Intention: {intent}\n"
        "Record: {record}\n"
        "Structure: {structure}\n"
    )
    select_prompt_c = ChatPromptTemplate.from_messages([
        ("system", select_template_3),
    ])
    select_chain_open = select_prompt_c | llm | StrOutputParser()
    cmds_text_open = select_chain_open.invoke({
        "input": question,
        "intent": label,
        "record": high_level_result,
        "structure": tree
    })

    # step 7: extract file
    commands_3 = extract_commands(cmds_text_open)
    commands_3 = [c for c in commands_3 if is_safe(c)]
    context_open = ""
    for cmd in commands_3:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="mcp-gateway-registry" )
        context_open +=  "Ran: " + cmd + "\n" + "Found: "+  result.stdout + "\n------------\n"
    # step 8: generate
    generate_templete = (
        "You are an assistant for a specific dataset 'mcp-gateway-registry' under the current directory. "
        "Use record of grep search as dataset to answer following question, say 'I don't know' if there is no relavent info in dataset"
        "Provide evidence from record with each claim if suitable in format"
        "Question: {input}\n"
        "Record: {record}\n"
    )
    generate_prompt = ChatPromptTemplate.from_messages([
        ("system", generate_templete),
    ])

    generate_chain = generate_prompt | llm | StrOutputParser()
    generate_result = generate_chain.invoke({
        "input": question,
        "record": context_open,
    })
    return(generate_result)

In [389]:
# p1q1
q1_a = mcp_qa_agent("What Python dependencies does this project use?")
print(q1_a)

21:39:44 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:39:44,957] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:39:45 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:39:45,232] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:39:45 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:39:45,280] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:39:46 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:39:46,740] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:39:46 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:39:46,832] p93245 {utils.

Based on the provided record, the Python dependencies used by this project are listed in the `pyproject.toml` file. Here are the dependencies:

* `aiofiles>=24.1.0`
* `fastapi>=0.115.12`
* `itsdangerous>=2.2.0`
* `jinja2>=3.1.6`
* `mcp>=1.9.3`
* `pydantic>=2.11.3`
* `pydantic-settings>=2.0.0`
* `httpx>=0.27.0`
* `python-dotenv>=1.1.0`
* `python-multipart>=0.0.20`
* `uvicorn[standard]>=0.34.2`
* `faiss-cpu>=1.7.4`
* `sentence-transformers>=3.0.0`
* `websockets>=15.0.1`
* `scikit-learn>=1.3.0`
* `torch>=1.6.0`
* `huggingface-hub>=0.31.1`
* `bandit>=1.8.3`
* `langchain-mcp-adapters>=0.0.11`
* `langgraph>=0.4.3`
* `langchain-aws>=0.2.23`
* `pytz>=2025.2`
* `strands-agents>=0.1.6`
* `strands-agents-tools>=0.1.4`
* `pyjwt>=2.10.1`
* `typing-extensions>=4.8.0`
* `httpcore[asyncio]>=1.0.9`
* `pyyaml>=6.0.0`
* `langchain-anthropic>=0.3.17`
* `matplotlib>=3.10.5`
* `psutil>=6.1.0`
* `email-validator>=2.2.0`
* `aiohttp>=3.8.0`
* `rich>=13.0.0`
* `requests>=2.31.0`
* `cisco-ai-a2a-scanner @ git+ht

In [390]:
# p1q2
q2_a = mcp_qa_agent("What is the main entry point file for the registry service?")
print(q2_a)

21:39:59 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:39:59,493] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:39:59 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:39:59,667] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:39:59 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:39:59,711] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:40:01 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:40:01,386] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:40:02 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:02,082] p93245 {utils.

Based on the provided record, I found that the main entry point file for the registry service is not explicitly mentioned in the search results. However, I can make an educated guess based on the file names and the search results.

The file names suggest that `RegistryService.cs` might be the main entry point file, but the search results for `RegistryService.cs` do not contain any matches for the string "RegistryService". This could indicate that the main entry point file is not named `RegistryService.cs` or that the string "RegistryService" is not present in the file.

On the other hand, the search results for `RegistryServiceFactory.cs`, `RegistryServiceConfig.cs`, `RegistryServiceInterface.cs`, and `RegistryServiceImplementation.cs` do not contain any matches for the string "RegistryService", which suggests that these files might not be the main entry point files.

However, I don't have enough information to make a definitive conclusion. If I had to make a guess, I would say that th

In [391]:
# p1q3
q3_a = mcp_qa_agent("What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)")
print(q3_a)

21:40:22 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:22,884] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:40:23 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:40:23,145] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:40:23 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:23,202] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:40:26 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:40:26,268] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:40:26 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:26,284] p93245 {utils.

Based on the provided records, the following programming languages and file types are used in this repository:

1. **Python**: 
   - The `Dockerfile` uses Python as the base image (`FROM python:3.12-slim`).
   - The `pyproject.toml` file lists Python dependencies.
   - The `test_api.py`, `test_auth.py`, `test_database.py`, `test_migrations.py`, `test_processor.py`, `test_rate_limiter.py`, `test_retention.py`, and `test_validator.py` files are Python test files.
   - The `test.py` file is a Python test file.
   - The `generate_access_token.py`, `generate_tokens.py`, `token_refresher.py`, `metrics_client.py`, and `migrate.py` files are Python scripts.

2. **TypeScript**: 
   - There is no direct evidence of TypeScript usage in the provided records.

3. **YAML**: 
   - The `pyproject.toml` file uses YAML syntax for configuration.
   - The `build-config.yaml` file is a YAML configuration file.

4. **JSON**: 
   - The `package.json` file is a JSON file.

5. **Dockerfile**: 
   - The `Docker

In [392]:
# p1q4
q4_a = mcp_qa_agent("How does the authentication flow work, from token validation to user authorization?")
print(q4_a)

21:40:46 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:46,681] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:40:46 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:40:46,875] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:40:46 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:46,911] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:40:48 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:40:48,391] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:40:48 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:40:48,514] p93245 {utils.

Based on the provided records, here's an overview of the authentication flow from token validation to user authorization:

**Token Validation**

1. The client (CLI tool or AI coding assistant) sends a request to the MCP Gateway Registry with an Authorization header containing a self-signed JWT token.
2. The NGINX Gateway receives the request and forwards it to the Auth Server for validation.
3. The Auth Server checks the token issuer (iss) and validates it using the SECRET_KEY (HS256) if the issuer matches "mcp-auth-server".
4. If the issuer does not match, the Auth Server tries to validate the token using the IdP JWKS (RS256).
5. The Auth Server extracts the scopes from the token and validates the server/tool access.

**User Authorization**

1. After successful token validation, the Auth Server returns a 200 OK response with X-User headers containing the user's information.
2. The NGINX Gateway proxies the request to the MCP Server.
3. The MCP Server checks the user's permissions base

In [393]:
# p1q5
q5_a = mcp_qa_agent("What are all the API endpoints available in the registry service and what scopes do they require?")
print(q5_a)

21:41:06 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:06,461] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:41:06 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:41:06,820] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:41:06 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:06,860] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:41:08 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:41:08,578] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:41:08 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:08,649] p93245 {utils.

Based on the provided records, I can answer your question about the API endpoints available in the registry service and their required scopes.

From the record of `cat api_endpoints.yaml | head -n 300`, I found the following API endpoints:

- `/api/v1/registry/endpoint`
- `/api/v1/registry/service`
- `/api/v1/registry/instance`
- `/api/v1/registry/credential`
- `/api/v1/registry/token`

However, the scopes required for these endpoints are not explicitly mentioned in the `api_endpoints.yaml` file.

From the record of `cat scopes.yaml | head -n 300`, I found the following scopes:

- `read:registry`
- `write:registry`
- `delete:registry`
- `read:service`
- `write:service`
- `delete:service`
- `read:instance`
- `write:instance`
- `delete:instance`
- `read:credential`
- `write:credential`
- `delete:credential`
- `read:token`
- `write:token`
- `delete:token`

However, the scopes required for each API endpoint are not explicitly mentioned in the `scopes.yaml` file.

From the record of `cat re

In [394]:
# p1q6
q6_a = mcp_qa_agent("How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?")
print(q6_a)

21:41:24 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:24,418] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:41:24 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:41:24,741] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:41:24 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:24,779] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
21:41:27 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-02-26 21:41:27,201] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
21:41:27 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-02-26 21:41:27,213] p93245 {utils.

To add support for a new OAuth provider (e.g., Okta) to the authentication system, we would need to modify the following files and implement the required interfaces.

1. **oauth2_providers.yml**: This file contains a list of supported OAuth providers. We would need to add a new entry for Okta, specifying the provider's name, client ID, client secret, and authorization URL.

```yml
# oauth2_providers.yml
- name: Okta
  client_id: <okta_client_id>
  client_secret: <okta_client_secret>
  authorization_url: https://<okta_domain>/oauth2/v2.0/authorize
```

2. **providers/base.py**: This file defines the base class for OAuth providers. We would need to create a new class that inherits from this base class and implements the required methods.

```python
# providers/base.py
class OAuthProvider:
    def __init__(self, client_id, client_secret, authorization_url):
        self.client_id = client_id
        self.client_secret = client_secret
        self.authorization_url = authorization_url

   

In [396]:
questions = [
    ("What Python dependencies does this project use?", q1_a),
    ("What is the main entry point file for the registry service?", q2_a),
    ("What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)", q3_a),
    ("How does the authentication flow work, from token validation to user authorization?", q4_a),
    ("What are all the API endpoints available in the registry service and what scopes do they require?", q5_a),
    ("How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?", q6_a),
]

output_lines = []

for i, (question, answer) in enumerate(questions, start=1):
    output_lines.append(f"Question {i}")
    output_lines.append("-" * 60)
    output_lines.append(f'"{question}"')
    output_lines.append("")
    output_lines.append("Answer:")
    output_lines.append(str(answer).strip())
    output_lines.append("\n\n")

final_text = "\n".join(output_lines)

with open("part1_results.txt", "w", encoding="utf-8") as f:
    f.write(final_text)

print("Saved to part1_results.txt")

Saved to part1_results.txt


# PART 2

### step1: classify query

In [511]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# step 1: classify
classify_template = (
    "You are an assistant for a specific dataset in 'data' directory. "
    "Classify the incoming question to determine what type of following 3 info is need. "
    "Then specify which file in structure of directory followed is need possibly"

    "Here is the structure of dataset: "
    "{structure}"

    "You MUST follow this response structure exactly:"

    "Summary of idea:"
    "<short explanation>"

    "Required files:\n"
    "<one or more lines, each a file path copied EXACTLY from the DATASET STRUCTURE; "
    "every path MUST start with 'data/' and MUST appear verbatim in the structure. "
    "If no file is needed or you are unsure, output: NONE>\n\n"

    "Sequence action:"
    "<ONE list only using pandas and/or context>"

    "Rules:"
    "- File paths MUST start with data/"
    "- Sequence action MUST be a Python-style list"
    "- Allowed values: pandas, context"
    "- Do NOT add extra sections or explanations"

    "At the end of your response, you MUST output a list describing the required action sequence."
    "The list may contain ONLY the following actions: pandas, context."
    "Rules:"
    "If only pandas is needed, output: [pandas]"
    "If pandas is followed by context, output: [pandas, context]"
    "If only context is needed, output: [context]"
    "Output ONLY the list. Do NOT add explanations, text, or formatting."
    "1. Analytical/numerical → CSV (use bash: grep, awk, cut)"
    "2. Product details/reviews → Unstructured text"
    "3. Both → Combine sources"

    "Question: {input}"
)
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", classify_template),
])

classifier_chain = classify_prompt | llm | StrOutputParser()

In [512]:
tree = subprocess.run("tree .", shell=True, capture_output=True, text=True, cwd="data" )
question = "What was the total revenue for Electronics category in December 2024?"
label = classifier_chain.invoke({"input": question, "structure": tree})
print(label)

01:43:59 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 01:43:59,507] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:43:59 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 01:43:59,908] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Summary of idea: Analyzing numerical data for a specific category and date.

Required files:
data/structured/daily_sales.csv

Sequence action: [pandas]


### step 2: act in sequence

In [513]:
import re

def parse_bracket_list(s: str) -> list[str]:
    m = re.search(r"\[(.*?)\]", s, flags=re.DOTALL)
    if not m:
        return []

    content = m.group(1).strip()
    if not content:
        return []

    items = content.split(",")

    return items

seq_ls = parse_bracket_list(label)
print(seq_ls)

['pandas']


In [517]:

# step 2: query
task_template = (
    "You are an assistant for a specific dataset in 'data' directory. "
    "This is previous progress on task: {label}"

    "You MUST output EXACTLY ONE of the following, depending on task.\n"

    "If do task == 'pandas', generate a temp_0.py with pandas code which read specific information needed in question," 
    "and save needed information to temp_0.txt in python code, the whole code should be wraped in ''' blanket"
    "DataColumns: date,product_id,product_name,category,units_sold,unit_price,total_revenue,region"

    "Else If do task == 'context', generate kernel commands which open and copy ALL text or info needed (DO NOT grep, etc) and stack into file temp_1.txt, command seperate by space."
    ", command string(s) as plain text (no markdown, no backticks, no code fences). Do NOT write the word 'bash', 'shell', or 'python'."

    "generate python code OR kernel commands (NOT BOTH) base on given task DIRECTLY, DO NOT explain or else! Do not combine both\n"

    "Do task = {task} base on question {input} now! DO NOT EXPLAIN"

)
task_prompt = ChatPromptTemplate.from_messages([
    ("system", task_template),
])

task_chain = task_prompt | llm | StrOutputParser()

In [518]:

label_2 = task_chain.invoke({"input": question, "label": label, "task": seq_ls[0]})
print(label_2)

01:45:07 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 01:45:07,242] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
01:45:07 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 01:45:07,802] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


```python
# temp_0.py
import pandas as pd

# Read the daily sales data
data = pd.read_csv('data/structured/daily_sales.csv')

# Filter the data for Electronics category in December 2024
filtered_data = data[(data['category'] == 'Electronics') & (data['date'].str.contains('2024-12'))]

# Calculate the total revenue for the filtered data
total_revenue = filtered_data['total_revenue'].sum()

# Save the result to a text file
with open('temp_0.txt', 'w') as f:
    f.write(f'Total revenue for Electronics category in December 2024: {total_revenue}')
```


In [526]:
from pathlib import Path
import subprocess
import sys
import textwrap


def clean_python_code(code: str) -> str:
    code = code.strip()

    # remove ```python ... ```
    code = re.sub(r"^```python\s*", "", code, flags=re.IGNORECASE)
    code = re.sub(r"^```", "", code)
    code = re.sub(r"```$", "", code)

    # remove '''python ... '''
    code = re.sub(r"^'''python\s*", "", code, flags=re.IGNORECASE)
    code = re.sub(r"^'''", "", code)
    code = re.sub(r"'''$", "", code)

    return code.strip()

label_2_ls = [] 

for task in seq_ls:
    if task == "pandas":
        code = task_chain.invoke({"input": question, "label": label, "task": task})
        label_2_ls.append(code)
        code = clean_python_code(code)
        temp_path = Path("temp.py")
        temp_path.write_text(textwrap.dedent(code), encoding="utf-8")
        result = subprocess.run(
            [sys.executable, "temp.py"],
            capture_output=True,
            text=True,
        )
    else: 
        commands_text= task_chain.invoke({"input": question, "label": label, "task": task})
        label_2_ls.append(commands)
        commands = [c.strip() for c in commands_text.splitlines() if c.strip()]
        for cmd in commands:
            print(cmd)
            result = subprocess.run(
                cmd,
                shell=True, 
                text=True,
                capture_output=True
            )

        


02:01:16 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:01:16,903] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:01:17 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:01:17,215] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


### step 3: answer base on data result

In [531]:

# step 3: summary

summary_template = (
    "You are an assistant for a specific dataset in 'data' directory. "
    "This is previous progress on task: {label}"

    "With given info of {info0} get by pandas query of {c0} and {info1} get by shell query {c1} from previous step,"

    "Answer the question: {question}"

)
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", summary_template),
])

summary_chain = summary_prompt | llm | StrOutputParser()

In [532]:
info0, info1 = "None", "None"
c0, c1 = "None", "None"
for task in seq_ls:
    if task == "pandas":
        with open("temp_0.txt") as f:
            info0 = f.read()
        c0 = label_2_ls[seq_ls.index(task)]
    elif task == "context":
        with open("temp_1.txt") as f:
            info1 = f.read()
        c1 = label_2_ls[seq_ls.index(task)]

        

label_3 = summary_chain.invoke({"question": question, "label": label, "info0": info0, "info1": info1, "c0":c0, "c1": c1})
print(label_3)

02:05:07 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:05:07,077] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:05:07 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:05:07,275] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


The total revenue for Electronics category in December 2024 was 142864.31.


In [567]:
    
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import subprocess
import json

def clean_python_code(code: str) -> str:
        code = code.strip()

        # remove ```python ... ```
        code = re.sub(r"^```python\s*", "", code, flags=re.IGNORECASE)
        code = re.sub(r"^```", "", code)
        code = re.sub(r"```$", "", code)

        # remove '''python ... '''
        code = re.sub(r"^'''python\s*", "", code, flags=re.IGNORECASE)
        code = re.sub(r"^'''", "", code)
        code = re.sub(r"'''$", "", code)

        return code.strip()

def parse_bracket_list(s: str) -> list[str]:
    m = re.search(r"\[(.*?)\]", s, flags=re.DOTALL)
    if not m:
        return []

    content = m.group(1).strip()
    if not content:
        return []

    items = content.split(",")

    return items


def mcp_agent_p2(question):
    
    # clean cache
    for f in ["temp_0.txt", "temp_1.txt", "temp.py"]:
        Path(f).unlink(missing_ok=True)

    # step 1: classify
    classify_template = (
        "You are an assistant for a specific dataset in 'data' directory. "
        "Classify the incoming question to determine what type of following 3 info is need. "
        "Then specify which file in structure of directory followed is need possibly"

        "Here is the structure of dataset: "
        "{structure}"

        "You MUST follow this response structure exactly:"

        "Summary of idea:"
        "<short explanation>"

        "Required files:\n"
        "<one or more lines, each a file path copied EXACTLY from the DATASET STRUCTURE; "
        "every path MUST start with 'data/' and MUST appear verbatim in the structure. "
        "If no file is needed or you are unsure, output: NONE>\n\n"

        "Sequence action:"
        "<ONE list only using pandas and/or context>"

        "Rules:"
        "- File paths MUST start with data/"
        "- Sequence action MUST be a Python-style list"
        "- Allowed values: pandas, context"
        "- Do NOT add extra sections or explanations"

        "At the end of your response, you MUST output a list describing the required action sequence."
        "The list may contain ONLY the following actions: pandas, context."
        "Rules:"
        "If only pandas is needed, output: [pandas]"
        "If pandas is followed by context, output: [pandas, context]"
        "If only context is needed, output: [context]"
        "Output ONLY the list. Do NOT add explanations, text, or formatting."
        "1. Analytical/numerical → CSV (use bash: grep, awk, cut)"
        "2. Product details/reviews → Unstructured text"
        "3. Both → Combine sources"

        "Question: {input}"
    )
    classify_prompt = ChatPromptTemplate.from_messages([
        ("system", classify_template),
    ])

    classifier_chain = classify_prompt | llm | StrOutputParser()
    tree = subprocess.run("tree .", shell=True, capture_output=True, text=True, cwd="data" )
    label = classifier_chain.invoke({"input": question, "structure": tree})

    seq_ls = parse_bracket_list(label)
    if len(seq_ls) >0:
        if "pandas" or "conetxt" in seq_ls:
            print("step 1 success")
        else:
            print("step 1 fail: wrong task")
    else:
        print("step 1 fail: no task extracted")
    # step 2: query
    task_template = (
        "You are an assistant for a specific dataset in 'data' directory. "
        "This is previous progress on task: {label}"

        "You MUST output EXACTLY ONE of the following, depending on task.\n"

        "If do task == 'pandas', generate a temp_0.py with pandas code which read specific information needed in question," 
        "and save needed information to temp_0.txt in python code, the whole code should be wraped in ''' blanket"
        "DataColumns: date,product_id,product_name,category,units_sold,unit_price,total_revenue,region"

        "Else If do task == 'context', generate kernel commands which open and copy ALL text or info needed (DO NOT grep, etc) and stack into file temp_1.txt, command seperate by space."
        ", command string(s) as plain text (no markdown, no backticks, no code fences). Do NOT write the word 'bash', 'shell', or 'python'."

        "generate python code OR kernel commands (NOT BOTH) base on given task DIRECTLY, DO NOT explain or else! Do not combine both\n"

        "Do task = {task} base on question {input} now! DO NOT EXPLAIN"

        "Optional: if following contain code and error message, means generation happened once, debug it for ouput code only!"
        "{old_code}, {error}"

    )
    task_prompt = ChatPromptTemplate.from_messages([
        ("system", task_template),
    ])

    task_chain = task_prompt | llm | StrOutputParser()

    label_2_ls = [] 

    old_code, error = "None", "None"
    for task in seq_ls:
        if task == "pandas":
            while not Path("temp_0.txt").exists():
                code = task_chain.invoke({"input": question, "label": label, "task": task, "old_code": old_code, "error":error})
                old_code = code
                code = clean_python_code(code)
                temp_path = Path("temp.py")
                temp_path.write_text(textwrap.dedent(code), encoding="utf-8")
                result = subprocess.run(
                    [sys.executable, "temp.py"],
                    capture_output=True,
                    text=True,
                )
                error = result
            old_code, error = "None", "None"
            label_2_ls.append(code)

        else: 
            commands_text= task_chain.invoke({"input": question, "label": label, "task": task, "old_code": old_code, "error":error})
            label_2_ls.append(commands_text)
            commands = [c.strip() for c in commands_text.splitlines() if c.strip()]
            for cmd in commands:
                print(cmd)
                result = subprocess.run(
                    cmd,
                    shell=True, 
                    text=True,
                    capture_output=True
                )

    # step 3: summary

    summary_template = (
        "You are an assistant for a specific dataset in 'data' directory. "
        "This is previous progress on task: {label}"

        "With given info of {info0} get by pandas query of {c0} and {info1} get by shell query {c1} from previous step,"

        "Answer the question: {question}"

    )
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", summary_template),
    ])

    summary_chain = summary_prompt | llm | StrOutputParser()
    
    info0, info1 = "None", "None"
    c0, c1 = "None", "None"
    for task in seq_ls:
        if task == "pandas":
            with open("temp_0.txt") as f:
                info0 = f.read()
            c0 = label_2_ls[seq_ls.index(task)]
        elif task == "context":
            with open("temp_1.txt") as f:
                info1 = f.read()
            c1 = label_2_ls[seq_ls.index(task)]

            

    generate_result = summary_chain.invoke({"question": question, "label": label, "info0": info0, "info1": info1, "c0":c0, "c1": c1})

    return(generate_result)

In [568]:
# p2q1
q2_a = mcp_agent_p2("What was the total revenue for Electronics category in December 2024?")
print(q2_a)

02:38:53 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:53,244] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:38:53 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:53,702] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:38:53 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:53,709] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:38:54 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:54,106] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:38:54 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:54,154] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:38:54 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:54,414] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:38:54 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:54,814] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:38:54 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:54,956] p93245 {utils.py:1630} INFO - Wr

Based on the previous step, the total revenue for the Electronics category in December 2024 was 142864.31.


In [569]:
# p2q2
q2_b = mcp_agent_p2("Which region had the highest sales volume?")
print(q2_b)

02:38:56 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:56,599] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:38:56 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:56,777] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:38:56 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:56,786] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:38:57 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:57,075] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:38:57 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:38:57,321] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:38:57 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:38:57,690] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Based on the previous progress, the region with the highest sales volume is Central. 

Here's the relevant code snippet from `temp_0.py`:

```python
# Find the region with the highest sales volume
max_revenue_region = region_revenue.loc[region_revenue['total_revenue'].idxmax()]

# Save the result to a text file
with open('temp_0.txt', 'w') as f:
    f.write(f"Region with the highest sales volume: {max_revenue_region['region']}\n")
    f.write(f"Total revenue: {max_revenue_region['total_revenue']}")
```

This code finds the region with the highest total revenue by using the `idxmax` method on the `total_revenue` column of the `region_revenue` DataFrame. The result is then saved to a text file named `temp_0.txt`. 

The output of this code is:

```
Region with the highest sales volume: Central
Total revenue: 394807.59
```


In [570]:
# p2q3
q2_c = mcp_agent_p2("What are the key features of the Wireless Bluetooth Headphones?")
print(q2_c)

02:39:03 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:03,392] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:39:03 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:03,836] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:03 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:03,839] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:39:04 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:04,139] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:04 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:04,278] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


```bash
cat data/unstructured/BEAU001_product_page.txt >> temp_1.txt
cat data/unstructured/BOOK001_product_page.txt >> temp_1.txt
cat data/unstructured/CLTH001_product_page.txt >> temp_1.txt
cat data/unstructured/ELEC001_product_page.txt >> temp_1.txt
cat data/unstructured/FOOD001_product_page.txt >> temp_1.txt
cat data/unstructured/HOME003_product_page.txt >> temp_1.txt
cat data/unstructured/OFFC001_product_page.txt >> temp_1.txt
cat data/unstructured/PETS001_product_page.txt >> temp_1.txt
cat data/unstructured/SPRT001_product_page.txt >> temp_1.txt
cat data/unstructured/TOYS001_product_page.txt >> temp_1.txt
```
This will copy all the text from the product pages into a single file called `temp_1.txt`.


02:39:04 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:04,972] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


The key features of the Wireless Bluetooth Headphones are:

- Active Noise Cancellation (ANC) with transparency mode
- Bluetooth 5.2 for stable connectivity up to 30ft range
- 40-hour battery life, quick charge (10 min = 3 hours playback)
- Premium memory foam ear cushions for all-day comfort
- Foldable design with carrying case included
- Built-in microphone for calls and voice assistant


In [571]:
# p2q4
q2_d = mcp_agent_p2("What do customers say about the Air Fryer's ease of cleaning?")
print(q2_d)

02:39:07 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:07,867] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:39:08 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:08,238] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:08 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:08,241] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:39:08 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:08,930] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


```bash
cat data/unstructured/ELEC001_product_page.txt >> temp_1.txt
cat data/unstructured/BEAU001_product_page.txt >> temp_1.txt
cat data/unstructured/BOOK001_product_page.txt >> temp_1.txt
cat data/unstructured/CLTH001_product_page.txt >> temp_1.txt
cat data/unstructured/FOOD001_product_page.txt >> temp_1.txt
cat data/unstructured/HOME003_product_page.txt >> temp_1.txt
cat data/unstructured/OFFC001_product_page.txt >> temp_1.txt
cat data/unstructured/PETS001_product_page.txt >> temp_1.txt
cat data/unstructured/SPRT001_product_page.txt >> temp_1.txt
cat data/unstructured/TOYS001_product_page.txt >> temp_1.txt
```
```python
import pandas as pd
# Read the product page text files into a list
files = ['data/unstructured/BEAU001_product_page.txt',


02:39:09 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:09,552] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


'data/unstructured/BOOK001_product_page.txt',
'data/unstructured/CLTH001_product_page.txt',
'data/unstructured/ELEC001_product_page.txt',
'data/unstructured/FOOD001_product_page.txt',
'data/unstructured/HOME003_product_page.txt',
'data/unstructured/OFFC001_product_page.txt',
'data/unstructured/PETS001_product_page.txt',
'data/unstructured/SPRT001_product_page.txt',
'data/unstructured/TOYS001_product_page.txt']
# Initialize an empty list to store the product information
product_info = []
# Loop through each file and extract the product information
for file in files:
with open(file, 'r') as f:
text = f.read()
# Assuming the product information is in the format "Product Name: <name>"
product_name = text.split(': ')[1]
# Assuming the product category is in the format "Category: <category>"
category = text.split(': ')[1]
# Assuming the product ease of cleaning is in the format "Ease of cleaning: <ease of cleaning>"
ease_of_cleaning = text.split(': ')[1]
product_info.append([product_name, ca

02:39:11 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:11,730] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Based on the provided product page text files, we can see that the Air Fryer 5.5L has the following information:

"Easy to clean with damp cloth."

However, to get a more comprehensive answer, let's analyze the customer reviews for the Air Fryer 5.5L:

Review 1 - Patricia W. (Verified Purchase) - 5 stars
"This air fryer has completely changed how I cook! Made the crispiest french fries I've ever had at home. My kids now prefer my homemade chicken nuggets over fast food. Easy to clean and the presets work perfectly."

Review 2 - Robert H. (Verified Purchase) - 5 stars
"Finally replaced my old deep fryer and couldn't be happier. Everything comes out crispy without the guilt. Made salmon last night and it was restaurant quality. The 5.5L size is perfect for our family of 4. Cleaning is a breeze!"

Review 3 - Lisa G. (Verified Purchase) - 4 stars
"Great product overall. Takes a bit of learning to get temperatures right for different foods. Wish the cord was longer - had to rearrange my cou

In [572]:
# p2q5
q2_e = mcp_agent_p2("Which product has the best customer reviews and how well is it selling?")
print(q2_e)

02:39:14 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:14,985] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:39:15 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:15,407] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:15 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:15,417] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:39:16 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:16,048] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:16 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:16,434] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:39:16 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:16,649] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:39:16 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:39:16,670] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


cat data/unstructured/BEAU001_product_page.txt data/unstructured/BOOK001_product_page.txt data/unstructured/CLTH001_product_page.txt data/unstructured/ELEC001_product_page.txt data/unstructured/FOOD001_product_page.txt data/unstructured/HOME003_product_page.txt data/unstructured/OFFC001_product_page.txt data/unstructured/PETS001_product_page.txt data/unstructured/SPRT001_product_page.txt data/unstructured/TOYS001_product_page.txt > temp_1.txt


02:39:17 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:39:17,837] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Based on the previous progress, the product with the best customer reviews is `BEAU001` with an average review of 2.1012006861063464. However, we need to verify this information.

To answer the question, we need to check the product details and sales data. Let's analyze the data:

```python
import pandas as pd

# Read daily sales data
sales_data = pd.read_csv('data/structured/daily_sales.csv')

# Read product details from text files
product_details = {}
for file in ['BEAU001_product_page.txt', 'BOOK001_product_page.txt', 'CLTH001_product_page.txt', 'ELEC001_product_page.txt', 'FOOD001_product_page.txt', 'HOME003_product_page.txt', 'OFFC001_product_page.txt', 'PETS001_product_page.txt', 'SPRT001_product_page.txt', 'TOYS001_product_page.txt']:
    with open(f'data/unstructured/{file}', 'r') as f:
        product_details[file.split('_')[0]] = f.read()

# Merge product details with sales data
merged_data = pd.DataFrame()
for product_id, details in product_details.items():
    product_data 

In [573]:
# p2q6
q2_f = mcp_agent_p2("I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?")
print(q2_f)

02:40:09 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:40:09,570] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:40:09 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:40:09,806] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:40:09 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:40:09,813] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


step 1 success


02:40:10 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:40:10,236] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:40:10 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:40:10,698] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
02:40:10 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:40:10,823] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler
02:40:10 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-02 02:40:10,844] p93245 {utils.py:3895} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


cat data/unstructured/SPRT001_product_page.txt data/structured/daily_sales.csv > temp_1.txt


02:40:11 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
[2026-03-02 02:40:11,305] p93245 {utils.py:1630} INFO - Wrapper: Completed Call, calling success_handler


Based on the analysis of the product details and sales data, I recommend the following product:

**Product Name:** [Product Name]
**Product ID:** [Product ID]
**Rating:** [Rating]
**Total Sales in West Region:** [Total Sales]

This product has a high rating of [Rating] and has sold a total of [Total Sales] units in the West region, making it a highly recommended product for fitness enthusiasts in this region.

Here's a summary of the top 3 recommended products:

1. **Product Name:** [Product Name]
**Product ID:** [Product ID]
**Rating:** [Rating]
**Total Sales in West Region:** [Total Sales]
2. **Product Name:** [Product Name]
**Product ID:** [Product ID]
**Rating:** [Rating]
**Total Sales in West Region:** [Total Sales]
3. **Product Name:** [Product Name]
**Product ID:** [Product ID]
**Rating:** [Rating]
**Total Sales in West Region:** [Total Sales]

These products have been selected based on their high ratings and sales performance in the West region, making them the top recommendati

In [574]:
questions = [
    ("What was the total revenue for Electronics category in December 2024?", q2_a),
    ("Which region had the highest sales volume?", q2_b),
    ("What are the key features of the Wireless Bluetooth Headphones?", q2_c),
    ("What do customers say about the Air Fryer's ease of cleaning?", q2_d),
    ("Which product has the best customer reviews and how well is it selling?", q2_e),
    ("I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?", q2_f),
]

output_lines = []

for i, (question, answer) in enumerate(questions, start=1):
    output_lines.append(f"Question {i}")
    output_lines.append("-" * 60)
    output_lines.append(f'"{question}"')
    output_lines.append("")
    output_lines.append("Answer:")
    output_lines.append(str(answer).strip())
    output_lines.append("\n\n")

final_text = "\n".join(output_lines)

with open("part2_results.txt", "w", encoding="utf-8") as f:
    f.write(final_text)

print("Saved to part2_results.txt")

Saved to part2_results.txt
